# 02. Risk model — учебный каркас

Цель: прогноз `TargetDPD30_12M` на момент выдачи кредита. Не используем `City` и `Region` в базовой risk-модели; изучаем их только для мониторинга ошибок.

**Задача для самостоятельной работы:** сначала Logistic Regression как baseline, затем сравнить с деревьями.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

ROOT = Path.cwd()
if not (ROOT / "data").exists(): ROOT = ROOT.parent
all_rows = pd.read_csv(ROOT / 'data/processed/model_dataset.csv')
df = all_rows[all_rows.IsMatureTarget.eq(1)].copy()
NUM = ['AgeAtIssue','IncomeMean6M_KGS','IncomeCV6M','CashflowMargin6M',
       'IncomeTrend3v6M','AppActiveDays6M','AgreementSumN','Rate','LoanTermMonths',
       'MonthlyPaymentKGS','DebtToIncomeRatio','PreviousLoansCount','ActivePreviousLoansCount',
       'PreviousPaymentsObserved','MaxHistoricalDPD','DPD30CountHistory',
       'DPD30CountLast12M','LatePaymentRatio','MonthsSinceLastDPD30',
       'TotalPriorLoanAmountKGS','NoCreditHistory']
CAT = ['ProductFamily','ProductCode','Occupation']
features = NUM + CAT
train = df[df.TimeSplit.eq('train')]
valid = df[df.TimeSplit.eq('validation')]
test  = df[df.TimeSplit.eq('test')]
print('Train / Valid / Test:', len(train), len(valid), len(test))

## Baseline Logistic Regression

Критично: imputer, scaler и encoder обучаются только на `train`, а не на всём датасете.

In [ ]:
preprocessor = ColumnTransformer([
    ('num', Pipeline([('fill', SimpleImputer(strategy='median')),
                      ('scale', StandardScaler())]), NUM),
    ('cat', Pipeline([('fill', SimpleImputer(strategy='most_frequent')),
                      ('onehot', OneHotEncoder(handle_unknown='ignore'))]), CAT)
])
model = Pipeline([
    ('prep', preprocessor),
    ('clf', LogisticRegression(max_iter=800, class_weight=None))
])
model.fit(train[features], train.TargetDPD30_12M)
valid_pd = model.predict_proba(valid[features])[:, 1]
print('Validation ROC-AUC:', roc_auc_score(valid.TargetDPD30_12M, valid_pd))
print('Validation PR-AUC:', average_precision_score(valid.TargetDPD30_12M, valid_pd))
print('Validation Brier:', brier_score_loss(valid.TargetDPD30_12M, valid_pd))
print('Validation prevalence:', valid.TargetDPD30_12M.mean())
# TODO: Почему accuracy здесь — недостаточная метрика?

## Самостоятельное задание

- Добавь `RandomForestClassifier` и CatBoost, оцени на validation.
- Проверь calibration plot: можно ли считать предсказанную вероятность реальным PD?
- Проанализируй out-of-time тест только **после** выбора модели, не используя его для подбора гиперпараметров.
- Запусти ablation: только кредитные признаки; затем добавь финансовые 6M; сравни прирост.
- Добавь анализ ошибок по продуктам и городам, но **не делай** выводов о реальном риске жителей городов.
- Учти, что `TimeSplit` — учебная схема, а для строгого production backtest дополнительно потребуется label-availability cutoff и embargo.